# Phase 1 — LEAD + CARLA + Scenario Runner closed loop (end-to-end)

This notebook reproduces the **whole Phase 1 workflow**: run the LEAD pretrained agent
(`tfv6_resnet34`) closed-loop on a cut-in route through CARLA + the Bench2Drive leaderboard /
Scenario Runner, then score it. Companion docs: [`PHASE1_NOTES.md`](../PHASE1_NOTES.md) (run log),
[`PLAN_R171.md`](../PLAN_R171.md) (plan), [`lead_carla_concepts.md`](../lead_carla_concepts.md).

**Pipeline:**

```
start CARLA  →  health check  →  closed-loop eval  →  inspect scores/infractions  →  (optional) video
 (manual)        carla_status      python -m lead          checkpoint_endpoint.json       chase-cam mp4
```

**Result we reproduce** (Town05 ParkingCutIn, route `24759`): `score_composed = 100`, route 100% complete,
0 collisions, 0 exceptions in ~21.9 s game time.

> ⚠️ **Two cells are heavy and side-effecting** — they are marked **`[HEAVY]`**. They will not run
> until *you* execute them:
> - **Start CARLA** — launches the GPU server. Per the project rules, you control the server manually.
> - **Closed-loop eval** — spins up the full leaderboard run (~1 min wall time).
>
> The inspection / video cells are cheap and read artifacts already on disk.

## 0. Setup — project root & environment sanity

Everything runs from the repo root in the `lead` conda env. `LEAD_PROJECT_ROOT` is **required** by
`lead/__main__.py`. We point the notebook's CWD at the repo root so all relative paths below resolve.

In [1]:
import json
import os
import subprocess
from pathlib import Path

# Resolve repo root (this notebook lives in <root>/notebooks/) and cd into it.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
os.environ["LEAD_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Phase 1 parameters.
CHECKPOINT = "outputs/checkpoints/tfv6_resnet34"
ROUTE = "data/benchmark_routes/bench2drive/24759.xml"  # Town05 ParkingCutIn
ROUTE_ID = Path(ROUTE).stem
EVAL_DIR = Path("outputs/local_evaluation") / ROUTE_ID

# CARLA connection. Host port 2000 is held by a foreign 0.9.16 server (another
# container under --network=host); run OUR 0.9.15 server on a free port instead.
PORT = 2100      # world port; streaming port is PORT+1 (2101)
TM_PORT = 8100   # traffic-manager port (default 8000 may also be taken)

print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"LEAD_PROJECT_ROOT : {os.environ['LEAD_PROJECT_ROOT']}")
print(f"checkpoint exists : {Path(CHECKPOINT).is_dir()}  ({CHECKPOINT})")
print(f"route exists      : {Path(ROUTE).is_file()}  ({ROUTE})")
print(f"eval output dir   : {EVAL_DIR}")
print(f"CARLA port        : {PORT}  (TM {TM_PORT}, streaming {PORT + 1})")

PROJECT_ROOT      : /workspace/lead
LEAD_PROJECT_ROOT : /workspace/lead
checkpoint exists : True  (outputs/checkpoints/tfv6_resnet34)
route exists      : True  (data/benchmark_routes/bench2drive/24759.xml)
eval output dir   : outputs/local_evaluation/24759
CARLA port        : 2100  (TM 8100, streaming 2101)


## 1. `[HEAVY]` Start CARLA  — *you run this manually*

`scripts/start_carla.sh` launches the headless server (`-RenderOffScreen`, no GUI). When started as
root it `setpriv`-drops to the `carla` user while keeping an ambient `CAP_DAC_OVERRIDE` so NVIDIA's
Vulkan driver can read the GPU's root-only PCI BAR files — without it CARLA falls back to CPU
rendering and segfaults. First boot on a fresh GPU spends a few minutes compiling shaders before the
port opens.

Run this in a terminal (recommended, so it owns the process) **or** execute the cell:

```bash
bash scripts/start_carla.sh 2100        # world port 2100 (streaming 2101)
```

> The server is detached (`setsid`) and survives the launching shell. `start_carla.sh` now **refuses
> to launch if the target port is already bound** (clean error instead of a bind segfault). Host port
> 2000 is held by a foreign **0.9.16** server in another container, so we use free port **2100** here.

In [2]:
# [HEAVY] Launches the GPU server. Uncomment to start from the notebook; logs -> /tmp/carla_<PORT>.log
# Prefer running `bash scripts/start_carla.sh <PORT>` in a terminal so the server isn't tied to the kernel.

# !bash scripts/start_carla.sh {PORT}
print(f"Skipped by default. Start CARLA on port {PORT}:  bash scripts/start_carla.sh {PORT}")

Skipped by default. Start CARLA on port 2100:  bash scripts/start_carla.sh 2100


## 2. Health check — is the server actually responsive?

`scripts/carla_status.sh` checks the process, the RPC port, a **real client handshake** (the source
of truth — a listening port can still belong to a wedged engine), and the GPU footprint.

> Note: `ss`/`netstat` report false negatives under `docker --network=host`; the script uses a
> `/dev/tcp` connect test + handshake instead. Re-run this until `responsive: YES` before evaluating.

In [3]:
ret = subprocess.run(["bash", "scripts/carla_status.sh", str(PORT)], text=True)
print(f"\n-> {'READY' if ret.returncode == 0 else 'NOT READY — do not run the eval yet'}")

CARLA status (port 2100)
  process  : RUNNING  (  10480       03:10 CarlaUE4-Linux-)
  port 2100: UP
  responsive: YES — client 0.9.15, server 0.9.15, map Carla/Maps/Town10HD_Opt
  gpu      : 10480, /opt/carla/CarlaUE4/Binaries/Linux/CarlaUE4-Linux-Shipping, 5783 MiB

-> READY


## 3. `[HEAVY]` Closed-loop evaluation

`python -m lead --bench2drive` is the wrapper: it builds `PYTHONPATH` for
`3rd_party/Bench2Drive/{leaderboard,scenario_runner}` and runs the leaderboard evaluator as a
subprocess (`scripts/eval_bench2drive.sh` is the lower-level equivalent). The leaderboard loads the
route's town, spawns the scenario (ParkingCutIn) + traffic, and drives the ego with LEAD's
`sensor_agent.py` in closed loop — sensors in, trajectory out, every tick — until the route completes
or fails.

Artifacts land in `outputs/local_evaluation/24759/`. Takes ~1 min wall time.

In [4]:
# [HEAVY] Full closed-loop run. Streams the leaderboard log live. Needs CARLA responsive (step 2).
cmd = [
    "python", "-m", "lead",
    "--checkpoint", CHECKPOINT,
    "--routes", ROUTE,
    "--bench2drive",
    "--port", str(PORT),
    "--traffic-manager-port", str(TM_PORT),
]
print("LEAD_PROJECT_ROOT=%s \\\n  %s\n" % (os.environ["LEAD_PROJECT_ROOT"], " ".join(cmd)))

# Uncomment to actually run:
# !LEAD_PROJECT_ROOT={os.environ['LEAD_PROJECT_ROOT']} python -m lead --checkpoint {CHECKPOINT} --routes {ROUTE} --bench2drive --port {PORT} --traffic-manager-port {TM_PORT}

LEAD_PROJECT_ROOT=/workspace/lead \
  python -m lead --checkpoint outputs/checkpoints/tfv6_resnet34 --routes data/benchmark_routes/bench2drive/24759.xml --bench2drive --port 2100 --traffic-manager-port 8100



## 4. Inspect results — scores

`checkpoint_endpoint.json` holds the leaderboard verdict. The driving score (`score_composed`) is
`score_route × score_penalty`: 100% route completion, no penalties → 100.

In [5]:
endpoint = json.loads((EVAL_DIR / "checkpoint_endpoint.json").read_text())
record = endpoint["_checkpoint"]["records"][0]
scores = record["scores"]
meta = record["meta"]

print(f"route id        : {record['route_id']}")
print(f"status          : {record['status']}")
print(f"driving score   : {scores['score_composed']}")
print(f"route completion: {scores['score_route']} %")
print(f"penalty mult.   : {scores['score_penalty']}")
print(f"duration (game) : {meta.get('duration_game')} s")
print(f"duration (sys)  : {meta.get('duration_system')} s")

route id        : RouteScenario_24759_rep0
status          : Completed
driving score   : 100.0
route completion: 100 %
penalty mult.   : 1.0
duration (game) : 21.9 s
duration (sys)  : 56.065 s


In [6]:
# Per-type infraction counts. Anything non-empty is a flagged event;
# `min_speed_infractions` is a soft metric that does NOT penalize score_composed.
for kind, events in record["infractions"].items():
    flag = "  <-- flagged" if events else ""
    print(f"{kind:32s}: {len(events)}{flag}")

collisions_layout               : 0
collisions_pedestrian           : 0
collisions_vehicle              : 0
red_light                       : 0
stop_infraction                 : 0
outside_route_lanes             : 0
min_speed_infractions           : 20  <-- flagged
yield_emergency_vehicle_infractions: 0
scenario_timeouts               : 0
route_dev                       : 0
vehicle_blocked                 : 0
route_timeout                   : 0


## 5. Inspect results — infractions & per-frame metrics

`infractions.json` and `metric_info.json` carry the detail behind the score (per-frame ego state,
events). Useful for diagnosing *why* a route failed when the score isn't 100.

In [7]:
for name in ("infractions.json", "metric_info.json"):
    p = EVAL_DIR / name
    if not p.is_file():
        print(f"{name}: (missing)")
        continue
    data = json.loads(p.read_text())
    if isinstance(data, dict):
        print(f"{name}: dict, keys = {list(data)[:12]}")
    elif isinstance(data, list):
        print(f"{name}: list of {len(data)} frames; frame[0] keys = {list(data[0])[:12] if data else []}")

infractions.json: dict, keys = ['infractions', 'video_fps']
metric_info.json: dict, keys = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']


## 6. (Optional) Record a chase-cam video

`scripts/carla_record_video.py [seconds] [n_traffic]` spawns an autopilot ego + traffic into the
currently-loaded map, attaches a chase camera, ticks in synchronous mode, and pipes frames through
ffmpeg to `outputs/snapshots/carla_<map>_<timestamp>.mp4`. This is a *visual confirmation that the
headless server is really rendering on the GPU* — it does not replay the eval route.

> A client that sets synchronous mode must restore async on exit, or the world is left frozen for the
> next client; the script handles that. Needs CARLA responsive (step 2).

In [10]:
# Uncomment to record ~10 s with 30 traffic vehicles:
# !python scripts/carla_record_video.py 10 30

snaps = sorted(Path("outputs/snapshots").glob("*.mp4"))
print("existing snapshots:")
for s in snaps:
    print(f"  {s}  ({s.stat().st_size // 1024} KB)")

existing snapshots:
  outputs/snapshots/carla_Town10HD_Opt_20260608_085906.mp4  (4581 KB)


In [11]:
# Inline-play the most recent snapshot (if any).
from IPython.display import Video

Video(str(snaps[-1]), embed=True, width=720) if snaps else print("No snapshot yet — record one above.")

## Summary

| Stage | Tool | Output |
| --- | --- | --- |
| Start server | `scripts/start_carla.sh` | headless CARLA on port 2000 (`/tmp/carla_2000.log`) |
| Health check | `scripts/carla_status.sh` | `responsive: YES` + GPU footprint |
| Closed-loop eval | `python -m lead --bench2drive` | `outputs/local_evaluation/24759/` |
| Score | `checkpoint_endpoint.json` | `score_composed=100`, route 100%, 0 collisions |
| Visual check | `scripts/carla_record_video.py` | `outputs/snapshots/*.mp4` |

**Goal #1 met** — LEAD + CARLA + Scenario Runner ran a full closed-loop cut-in scenario and scored it.

**Next (Phase 2):** the faithful R171 adjacent-lane case is `HighwayCutIn` (`2286`/`3072`), which are
**Town12** routes — not installed in the base CARLA tarball. They need `AdditionalMaps_0.9.15` (~4 GB)
extracted into `/opt/carla`. Until then, Town05 `ParkingCutIn`/`StaticCutIn` (`24759`/`26396`) cover
the cut-in family. See [`PLAN_R171.md`](../PLAN_R171.md) → Phase 2.